In [ ]:
import math
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import rasterio
import rasterio.plot  # submodule is not imported by `import rasterio` alone
import torch
import folium
import time
import networkx as nx

from pathlib import Path
from IPython.display import Image  # For displaying the training GIF
from rasterio.transform import Affine
from matplotlib.colors import LogNorm

# Use this to load the functions used in this notebook, and call them with `deepssf.function_name()`
import deepssf

print(f"deepssf {deepssf.__version__}")
print(f"torch {torch.__version__}")

# Setup the script

## Set random seed and the device for training the model

If you have a GPU, this is where it will be specified (automatically).

In [ ]:
# Reproducibility — fix random seeds so results are the same on every run.
torch.manual_seed(42)
np.random.seed(42)

# DEVICE is capitalised — module-level constant (set once, never reassigned).
# get_device() returns 'cuda' if a GPU is available, 'mps' on Apple Silicon,
# or 'cpu' otherwise.  All tensors and the model are moved to this device.
DEVICE = deepssf.get_device()
print(f"Device: {DEVICE}")

## Set script paths
If you have cloned/forked/downloaded the GitHub repo (https://github.com/swforrest/deepSSF_package) you shouldn't need to change these, but for your own analyses you will want to set these to point to where your data is and where you want outputs saved.

*All-caps names below are module-level constants — set once here and referenced throughout the notebook without being reassigned.  Python has no built-in constant keyword, so capitalisation is the community convention (PEP 8) to signal "do not change this value".*

In [ ]:
DATA_DIR = Path("../src/deepssf/datasets/data")  # location of the bundled sample data
CSV_PATH = DATA_DIR / "buffalo_djelk_id2005.csv"  # GPS tracking data, individual 2005
LAYER_PATHS = {
    "ndvi":  str(DATA_DIR / "ndvi_2005.tif"),   # Normalised Difference Vegetation Index
    "slope": str(DATA_DIR / "slope_2005.tif"),  # terrain slope derived from a DEM
}
OUTPUT_DIR   = Path("outputs")                   # figures and model checkpoints go here
SNAPSHOT_DIR = OUTPUT_DIR / "training_snapshots"  # per-batch prediction-map snapshots
OUTPUT_DIR.mkdir(exist_ok=True)

# Import data
The csv file is imported with a timestamp and an x and y coordinate. In our case the data was already in a projected format that is specific to Australia (EPSG:3112), which is the same crs as the raster data which will be imported.

In [ ]:
raw_df = pd.read_csv(CSV_PATH)
print(f"Raw GPS fixes : {len(raw_df):,}")
print(f"Individuals   : {raw_df['id'].nunique()}")

# Sort by the individual ID and timestamp to ensure correct temporal order
raw_df = raw_df.sort_values(by=["id", "time"]).reset_index(drop=True)

# # # Convert the 'time' column to the correct timezone (here Australia/Darwin) 
# raw_df["time"] = (
#         pd.to_datetime(raw_df["time"], utc=True)
#         .dt.tz_convert("Australia/Darwin")
#     )

raw_df.head()

## `load_environmental_layers()`

Reads each GeoTIFF listed in LAYER_PATHS, min–max

- Scales every band to [0, 1], and returns:
    * `env_layers`       — dict mapping layer name → 2-D NumPy array (H × W)
    * `raster_transform` — Affine object that converts pixel (row, col) indices to the projected coordinates (easting, northing) of the raster(s), which should be in the same CRS as the GPS data.  This transform is used whenever we need to translate between pixel space and real-world coordinates.


In [ ]:
env_layers, raster_transform = deepssf.load_environmental_layers(LAYER_PATHS)
print("Layers loaded:", list(env_layers.keys()))
for name, arr in env_layers.items():
    print(f"  {name}: shape={arr.shape}, min={arr.min():.3f}, max={arr.max():.3f}")

# Each band of a multi-band layer becomes one CNN input channel, so count bands
# (not layers) here and reuse the total when setting ModelParams below.  Deriving
# it means switching rasters in LAYER_PATHS needs no other edits.
N_SPATIAL_CHANNELS = sum(1 if arr.ndim == 2 else arr.shape[0] for arr in env_layers.values())
print(f"Spatial channels: {N_SPATIAL_CHANNELS}")

###  Plot each spatial covariate with the observed GPS locations overlaid.
This confirms the raster extent covers the extent of the tracking data and gives a sense of the landscape the model will learn from.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

layer_plot = [
    ("ndvi",  "NDVI (scaled 0–1)",  "YlGn"),
    ("slope", "Slope (scaled 0–1)", "viridis"),
]

for ax, (name, label, cmap) in zip(axes, layer_plot, strict=False):
    rasterio.plot.show(env_layers[name], transform=raster_transform, ax=ax, cmap=cmap)
    ax.plot(
        raw_df["x"], raw_df["y"],
        "r-o", alpha=0.5, markersize=1, linewidth=0.8, label="GPS trajectory",
    )
    ax.set_xlabel("Easting (m, projected)")
    ax.set_ylabel("Northing (m, projected)")
    ax.set_title(label)
    ax.legend(loc="upper right", markerscale=6)

plt.suptitle("Spatial covariates with observed GPS locations", y=1.01)
plt.tight_layout()
plt.savefig(str(OUTPUT_DIR / "spatial_covariates.png"), dpi=100)
plt.show()

## `prepare_movement_df()`
Converts a fixes dataframe (one row per GPS location) into a *step* dataframe (one row per movement between consecutive fixes).

- For each step it computes:
    * `dx`, `dy`         — displacement in projected CRS units (metres)
    * `bearing`          — direction of travel (radians; 0 = north, clockwise positive). These entries will also be used to calculate the bearing of the *previous* step, used as a model input so the network can capture directional persistence by estimating a turning angle
    * `dt_hour`          — time elapsed between consecutive fixes (hours)
    * `hour_t1`, `yday_t1` — decimal hour-of-day and day-of-year at the step start location
    * `*_sin1` / `*_cos1`  — sine/cosine encodings of hour and yday (see SCALAR_COLS below)

The trailing underscore on column names (x1_, y1_, t1_, etc.) flags them as "start of step" variables; x2_, y2_, t2_ are the corresponding "end of step" values, following the convention of the `amt` package [@Signer2019-fi]. The last fix for each individual is dropped because no forward step can be formed.

In [ ]:
# The CSV is already in amt step format, but prepare_movement_df() expects one row
# per *fix* and rebuilds the steps itself (it also derives bearing, dt_hour and the
# cyclic time encodings the model needs).  Point it at the step-start columns and
# let it re-form the steps from consecutive fixes.
step_df = deepssf.prepare_movement_df(
    raw_df, 
    id_col="id", 
    time_col="time", 
    x_col="x", 
    y_col="y",
    tz="Australia/Darwin"
)

n_dropped = len(raw_df) - len(step_df)
print(f"Movement steps : {len(step_df):,}  ({n_dropped} dropped — last per individual)")
step_df.head()

### Check the timezone

In [ ]:
step_df["t1_"].dt.tz

## Visualise some data summaries

Partly this is diagnostic, to check whether the time is in the correct timezone (we know that buffalo have high movement periods around dawn and dusk, so we can see whether that aligns). We can also check these summaries against summaries of the simulated trajectories to see whether we've captured these behaviours.

### Distribution of step lengths

In [ ]:
# Distribution of step lengths, with the median and mean marked
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(step_df["sl_"], bins=250, color="tab:blue", alpha=0.7)
ax.axvline(step_df["sl_"].median(), color="black", lw=2, label="Median")
ax.axvline(step_df["sl_"].mean(), color="black", lw=2, ls="--", label="Mean")
ax.set_xlabel("Step length (m)")
ax.set_ylabel("Count")
ax.set_xlim(0, 2000)
ax.legend()
plt.tight_layout()
plt.show()

### Log of step lengths

In [ ]:
# Distribution of log step lengths, with the median and mean marked
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(np.log(step_df["sl_"]), bins=250, color="tab:blue", alpha=0.7)
ax.axvline(np.log(step_df["sl_"].median()), color="black", lw=2, label="Median")
ax.axvline(np.log(step_df["sl_"].mean()), color="black", lw=2, ls="--", label="Mean")
ax.set_xlabel("Log step length (m)")
ax.set_ylabel("Count")
ax.legend()
plt.tight_layout()
plt.show()

### Step lengths through time

In [ ]:
# Step length by hour of day, with the hourly median and mean overlaid
hourly_sl_median = step_df.groupby(step_df["hour_t1"].round().mod(24))["sl_"].median()
hourly_sl_mean = step_df.groupby(step_df["hour_t1"].round().mod(24))["sl_"].mean()

fig, ax = plt.subplots(figsize=(8, 4))
ax.scatter(step_df["hour_t1"], step_df["sl_"], s=4, alpha=0.15, color="tab:blue", label="Steps")
ax.plot(hourly_sl_mean.index, hourly_sl_mean.values, color="red", lw=2, marker="s", ms=4, label="Hourly mean")
ax.plot(hourly_sl_median.index, hourly_sl_median.values, color="black", lw=2, marker="o", ms=4, label="Hourly median")
ax.set_xlabel("Hour of day")
ax.set_ylabel("Step length (m)")
ax.set_xticks(range(0, 25, 3))
ax.set_xlim(0, 24)
ax.set_ylim(0, 1000)
ax.legend()
plt.tight_layout()
plt.show()

### Distribution of turning angles

In [ ]:
# Distribution of turning angles (NaN for each individual's first step, so dropped)
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(step_df["ta_"].dropna(), bins=72, range=(-np.pi, np.pi), color="tab:orange", alpha=0.7)
ax.set_xlabel("Turning angle (radians)")
ax.set_ylabel("Count")
ax.set_xticks([-np.pi, -np.pi / 2, 0, np.pi / 2, np.pi])
ax.set_xticklabels(["-π", "-π/2", "0", "π/2", "π"])
ax.set_xlim(-np.pi, np.pi)
plt.tight_layout()
plt.show()

### Turning angles through time

As turning angles are circular, the mean/median sit
near 0 regardless of spread. Instead, we summarise each hour with the circular SD 
and the shortest arc containing 50% of the angles (which may wrap across ±π), i.e.,
where are the densest 50% of the turning angles concentrated?

From this plot, we can see that movements are most directed (mean at 0 and most TAs
near 0) around dawn and dusk, reflecting the longer steps in those periods. 
This shows that buffalo appear to be travelling in a directed way during those periods. 

In [ ]:
# create a helper function to summarise turning angles by hour of day 
# into circular SD and the shortest arc containing 50% of the angles
def circ_summary(theta, prob=0.5):
    """Circular SD and the shortest arc containing `prob` of the angles."""
    theta = np.sort(np.asarray(theta))
    n = len(theta)
    circ_sd = np.sqrt(-2 * np.log(np.abs(np.mean(np.exp(1j * theta)))))
    k = int(np.ceil(prob * n))
    ext = np.concatenate([theta, theta + 2 * np.pi])  # unwrap so arcs can cross ±π
    widths = ext[k - 1 : k - 1 + n] - ext[:n]
    i = np.argmin(widths)
    return pd.Series({"circ_sd": circ_sd, "arc_start": ext[i], "arc_end": ext[i + k - 1]})

ta_df = step_df.dropna(subset=["ta_"])
hourly_ta = ta_df.groupby(ta_df["hour_t1"].round().mod(24))["ta_"].apply(circ_summary).unstack()

fig, (ax, ax_sd) = plt.subplots(2, 1, figsize=(8, 6), sharex=True, height_ratios=[3, 1])
ax.scatter(ta_df["hour_t1"], ta_df["ta_"], s=4, alpha=0.15, color="tab:orange", label="Steps")
for hour, row in hourly_ta.iterrows():
    # draw the arc as one segment, or two if it wraps past π
    segments = [(row.arc_start, min(row.arc_end, np.pi))]
    if row.arc_end > np.pi:
        segments.append((-np.pi, row.arc_end - 2 * np.pi))
    for lo, hi in segments:
        ax.vlines(hour, lo, hi, color="black", lw=3, label="Densest 50%" if hour == hourly_ta.index[0] and lo == row.arc_start else None)
ax.set_ylabel("Turning angle (radians)")
ax.set_yticks([-np.pi, -np.pi / 2, 0, np.pi / 2, np.pi])
ax.set_yticklabels(["-π", "-π/2", "0", "π/2", "π"])
ax.set_ylim(-np.pi, np.pi)
ax.legend(loc="upper right")

ax_sd.plot(hourly_ta.index, hourly_ta["circ_sd"], color="black", lw=2, marker="o", ms=4)
ax_sd.set_ylabel("Circular SD")
ax_sd.set_xlabel("Hour of day")
ax_sd.set_xticks(range(0, 25, 3))
ax_sd.set_xlim(0, 24)
plt.tight_layout()
plt.show()

## `SCALAR_COLS`
These are non-spatial inputs. For processing with convolutional layers, they are converted into spatial windows (rasters/arrays), and stacked with the truly spatial covariates.

The model receives a fixed-size spatial window centred on each step's start location.  To let the network express time-varying habitat preferences (e.g. different nocturnal vs diurnal use), scalar covariates are tiled into additional channels of the same H × W size, so every pixel "knows" the current time context when habitat suitability is estimated.

- `hour_t1_sin1` / `hour_t1_cos1`
    * Sine and cosine of (2π × hour / 24), encoding time-of-day as a *circular* variable.  The circular encoding means 23:00 and 01:00 are treated as adjacent — a plain hour value would have an artificial discontinuity at midnight.

- `yday_t1_sin1` / `yday_t1_cos1`
    * Same circular encoding for day-of-year (1–365), capturing seasonal variation in habitat use (e.g. dry-season vs wet-season movements).

- `dt_hour`
    * Elapsed time between consecutive fixes (hours).  Because GPS collars do not always fix at a perfectly regular rate, including dt_hour lets the movement kernel rescale expected displacement proportionally to the actual time interval rather than assuming a fixed fix rate. Note: dt_hour must be supplied at *simulation* time as well, so that `simulate_trajectory()` produces inputs consistent with training.

In [ ]:
SCALAR_COLS = ["hour_t1_sin1",
               "hour_t1_cos1", 
               "yday_t1_sin1", 
               "yday_t1_cos1", 
               "dt_hour"]


## `PIXEL_SIZE`
Raster resolution in CRS units (metres here).  Read straight off `raster_transform`
rather than typed in: the transform is the authority on the grid the model is
trained against, and a hand-entered value that disagrees with it silently
mis-scales the movement kernel — which is parameterised in metres — against a
grid measured in pixels.

In [ ]:
# transform.a is the cell width, transform.e the (negative) cell height
PIXEL_SIZE = abs(raster_transform.a)

if not math.isclose(abs(raster_transform.a), abs(raster_transform.e)):
    raise ValueError(
        f"Non-square pixels ({abs(raster_transform.a)} x {abs(raster_transform.e)}); "
        "the model assumes square cells."
    )

print(f"Pixel size: {PIXEL_SIZE} m") 

## `WINDOW_SIZE`
The size of the local availability window in cells/pixels. This is what the model will crop out during training to define the **local availability window**.

The maximum step length that the model will allow is the distance between the centre cell 
and the nearest edge (in reality some steps may be longer if they are in the corners).

In [ ]:
WINDOW_SIZE = 101  # odd number of pixels so there is a central pixel

print(f"Pixel size: {PIXEL_SIZE} m  →  window {WINDOW_SIZE} px = "
      f"{WINDOW_SIZE * PIXEL_SIZE:,.0f} m across")

print(f"Maximum possible step: {((WINDOW_SIZE - 1) / 2) * PIXEL_SIZE:,.0f} m")

## `BATCH_SIZE`
Number of training samples (movement steps) per training mini-batch. From what I've found training seems to be fairly robust this (doesn't seem to change performace much), but I have not rigorously tested this.

In [ ]:
BATCH_SIZE  = 32

## `N_CONV_LAYERS_MOVE`
To set the number of convolutional layers in the movement subnetwork of the model.

Each of the convolution layers has a max pooling layer, so more layers will result in greater dimension reduction that will feed into the fully-connected layer that outputs the movement parameters.

In [ ]:
N_CONV_LAYERS_MOVE = 3

## `N_CONV_LAYERS_HAB`
To set the number of convolutional layers in the habitat selection subnetwork of the model.

These convolution layers do not have max pooling layers, so there is no dimension reduction. More layers will increase the depth of the model and allow for more abstracted features to be identified, but will slow model training/prediction.

This count includes the final convolution layer, which is the one that collapses the feature maps down to the single-channel habitat selection surface, so the minimum is 1.

In [ ]:
N_CONV_LAYERS_HAB = 4

## `OUTPUT_CHANNELS`
Number of convolutional filters (and therefore feature maps) for **each** convolution layer, for both the movement and habitat selection convolution layers. More convolutional filters can provide more flexible functions, but will increase the number of parameters and slow down training times.

In [ ]:
OUTPUT_CHANNELS = 8

## Other convolution layer inputs
These are pretty typical values, so I haven't investigated how these might change the performance of the model.

### `KERNEL_SIZE`
- Defines the size (in number of pixles) of the convolutional kernels. Larger convolutional kernels have a greater pixel-neighbourhood awareness.
### `KERNEL_SIZE_MP`
- Defines kernel size for max-pooling (if kernel size = stride, a kernel size of 2 will halve the dimensions on each side, resulting in 1/4 of the number of cells)
### `STRIDE_MP`
- Defines max-pooling stride size

In [ ]:
KERNEL_SIZE     = 3     # conv kernel spatial size
KERNEL_SIZE_MP  = 2     # max-pool kernel size
STRIDE_MP       = 2     # max-pool stride 

### Pre-calculate the flattened CNN output size
This value will be the input to the fully-connected layers in the movement subnetwork, and is calculated here so that ModelParams can be set without needing to hard-code it.

`flattened_conv_dim()` reproduces exactly what the layers inside `ConvJointModel` do to the spatial dimensions:
* conv (padding=1) keeps the spatial dimension unchanged at each layer
* maxpool k=2, s=2 halves the spatial dimension (floor division) at each layer

Because every movement conv layer is paired with a max pooling layer, this size depends on `N_CONV_LAYERS_MOVE` — so it has to be recalculated whenever that (or `WINDOW_SIZE`, or `OUTPUT_CHANNELS`) changes. It raises an error if the window is pooled away to nothing, which means too many layers for the window size.

In [ ]:
DENSE_DIM = deepssf.flattened_conv_dim(
    image_dim=WINDOW_SIZE,
    n_conv_layers_move=N_CONV_LAYERS_MOVE,
    output_channels=OUTPUT_CHANNELS,
)
print(f"After {N_CONV_LAYERS_MOVE}x conv+maxpool: dense_dim_in_all={DENSE_DIM}")

## `DENSE_DIM_HIDDEN`
How many weights ('neurons') should there be for the hidden layers in the movement kernel?

In [ ]:
DENSE_DIM_HIDDEN = 64 # hidden units in the scalar MLP

## `filter_steps_by_window()` 
Removes steps whose straight-line displacement exceeds half the spatial window (WINDOW_SIZE × PIXEL_SIZE / 2 metres). Such steps would place the true next location outside the predicted probability map, causing an index error in the loss function.

In [ ]:
original_data_size = len(step_df)

step_df = deepssf.filter_steps_by_window(
    step_df, window_size=WINDOW_SIZE, pixel_size=PIXEL_SIZE
)

print(f"Initial number of steps: {original_data_size:,}")
print(f"Steps after window filter: {len(step_df):,}")
print(f"Data reduction: {original_data_size - len(step_df):,} ({(original_data_size - len(step_df)) / original_data_size * 100:.1f}%)")

## `make_dataloaders()`
Splits step_df into train / val / test subsets (here 80 / 10 / 10 %) and returns a PyTorch DataLoader for each.

**The split is a `torch.utils.data.random_split`, not a chronological one** — each subset is a random sample of steps drawn from across the whole dataset and all 28 individuals. Consecutive steps are highly autocorrelated, so a random split leaks information between train and test and will make the validation numbers below look better than genuine out-of-sample performance. If you want an honest held-out estimate, split `step_df` yourself (by individual, or by time) and build the DataLoaders from the resulting subsets.

- Each DataLoader yields tuples of:
    * `x1`
        — spatial window tensor  (batch × n_spatial_channels × H × W)
        - One channel per raster *band* in LAYER_PATHS, cropped around each step's start pixel.
    * `x2`
        — scalar covariate tensor (batch × len(SCALAR_COLS))
        - Raw scalar values; broadcast into spatial maps *inside* the model.
    * `x3`
        — previous bearing tensor (batch × 1)
        - Conditions the movement kernel on the direction of the prior step.
    * `(px2, py2)`
        — pixel indices of the *true* next location within the window
        - Used by the loss function to look up the predicted probability at the observed destination.

The layer_paths argument causes the function to re-read the rasters internally, so the DataLoader is independent of the env_layers dict loaded above.

In [ ]:
dl_train, dl_val, dl_test = deepssf.make_dataloaders(
    layer_paths=LAYER_PATHS,
    window_size=WINDOW_SIZE,
    batch_size=BATCH_SIZE,
    train_split=0.8,
    val_split=0.1,
    scalar_cols=SCALAR_COLS,
    df=step_df,
)
print(f"Train batches : {len(dl_train)}")
print(f"Val   batches : {len(dl_val)}")
print(f"Test  batches : {len(dl_test)}")

### Inspect one batch to verify tensor shapes match ModelParams expectations
`x1` has one channel per spatial layer (ndvi + slope = 2).  The scalar covariates in `x2` are broadcast to full H × W maps *inside* ConvJointModel and concatenated with `x1` there, giving the CNN 2 + len(SCALAR_COLS) = 7 input channels total.

The `px2` and `py2` are the target that the model is trying to predict, and are the pixel values of the next step (with the centre cell being (50, 50))

*`next(iter(dl_train))` will call the next batch, which are randomly shuffled, so calling the function again will provide different training samples.*

In [ ]:
x1, x2, x3, (px2, py2), _ = next(iter(dl_train))
print(f"Spatial   x1 : {x1.shape}   (batch × channels × H × W)")
print(f"Scalars   x2 : {x2.shape}   (batch × len(SCALAR_COLS))")
print(f"Bearing   x3 : {x3.shape}   (batch × 1)")
print(f"Next-step px2: {px2[:4].tolist()}  py2: {py2[:4].tolist()}")

### Plot some samples from the first batch

Plot the spatial window "tiles" from the first batch: one row per sample, one column per raster band.  The centre pixel is the step's start location and the red cross marks the true next location that the loss function will be trying to predict (and used to evaluate the loss function).

In [ ]:
N_SHOW = 4  # how many samples from the batch to display

# Band labels straight from the GeoTIFF metadata, in the channel order the model sees
channel_labels = []
for name, arr in env_layers.items():
    if arr.ndim == 2:
        channel_labels.append(name)
        continue
    with rasterio.open(LAYER_PATHS[name]) as src:
        descs = src.descriptions
    for band in range(arr.shape[0]):
        channel_labels.append(descs[band] if descs and descs[band] else f"{name} b{band + 1}")

n_show = min(N_SHOW, x1.shape[0])
n_chan = x1.shape[1]
centre = WINDOW_SIZE // 2

fig, axes = plt.subplots(
    n_show, n_chan, figsize=(3.0 * n_chan, 3.0 * n_show), squeeze=False
)

for row in range(n_show):
    for col in range(n_chan):
        ax = axes[row, col]
        tile = x1[row, col].numpy()
        ax.imshow(tile, cmap="viridis", origin="upper")
        # Start location: always the centre cell of the window by construction
        ax.plot(centre, centre, "w+", markersize=9, markeredgewidth=1.5)
        # True next step — px2 is the column index, py2 the row index
        ax.plot(px2[row].item(), py2[row].item(), "rx", markersize=9, markeredgewidth=1.5)
        ax.set_xticks([])
        ax.set_yticks([])
        if row == 0:
            ax.set_title(channel_labels[col], fontsize=10)
        if col == 0:
            ax.set_ylabel(
                f"sample {row}\n→ ({px2[row].item()}, {py2[row].item()})", fontsize=8
            )

fig.suptitle(
    f"First batch: {WINDOW_SIZE}×{WINDOW_SIZE} tiles  "
    "(white + = step start, red × = observed next step)",
    y=1.0,
)
plt.tight_layout()
plt.savefig(str(OUTPUT_DIR / "batch_tiles.png"), dpi=100, bbox_inches="tight")
plt.show()


## `DROPOUT`
Dropout switches some proportion of the weights off during each forward pass of the model, to prevent the model becoming overly reliant on certain routes through the network, which could result in overfitting and limit generalisability. 

Here it only applies to the fully connected layers in the movement kernel, not for any of the convolution layers.

In [ ]:
DROPOUT_RATE = 0.2  # proportion of hidden units to drop out during training

## `ModelParams` 
A thin dict-like container (it is a wrapper around something else rather than storing data) that groups all architecture hyper-parameters so they can be passed to ConvJointModel and logged together.

Pretty much everything has been set above, and this is where they are stored and how they're provided to the model. The parameters that are defined here (stride, padding, num_movement_params) should be kept as they are to ensure the model works as expected. The others should be able to be changed, which will change the shape of the model. 


In [ ]:
params = deepssf.ModelParams({
    "batch_size":                BATCH_SIZE,
    "image_dim":                 WINDOW_SIZE,           # spatial window size (pixels)
    "pixel_size":                PIXEL_SIZE,            # metres per pixel
    "dim_in_nonspatial_to_grid": len(SCALAR_COLS),      # scalar cols tiled into window
    "dense_dim_in_nonspatial":   len(SCALAR_COLS),      # scalar inputs to the dense MLP
    "dense_dim_hidden":          DENSE_DIM_HIDDEN,                   # hidden units in the scalar MLP
    "dense_dim_in_all":          DENSE_DIM,             # flattened CNN output size
    # One channel per raster band (not per layer) + the tiled scalar covariates
    "input_channels":            N_SPATIAL_CHANNELS + len(SCALAR_COLS),
    "output_channels":           OUTPUT_CHANNELS,       # feature maps per conv layer
    "kernel_size":               KERNEL_SIZE,           # conv kernel spatial size
    "stride":                    1,                     # conv stride (should be kept at 1)
    "kernel_size_mp":            KERNEL_SIZE_MP,        # max-pool kernel size
    "stride_mp":                 STRIDE_MP,             # max-pool stride 
    "padding":                   1,                     # same-padding preserves spatial dimensions (should be kept at 1)
    "num_movement_params":       12,                    # movement kernel parameters (changing this will require modifying the movement kernel)
    "n_conv_layers_hab":         N_CONV_LAYERS_HAB,     # conv layers, habitat branch
    "n_conv_layers_move":        N_CONV_LAYERS_MOVE,    # conv+maxpool layers, movement
    "dropout":                   DROPOUT_RATE,                   # dropout probability (0.0 = no dropout, 0.2 = 20% dropout)
    "device":                    DEVICE,
})
print(f"input_channels: {params.input_channels}  (must match x1 channels + len(SCALAR_COLS))")

## `ConvJointModel` 
The deepSSF model! A joint habitat–movement model with two heads:
1. Habitat head 
    — a small CNN applied to the spatial window (env layers + tiled scalars) that outputs a per-pixel log-suitability map.
2. Movement head 
    — a dense MLP (fully-connected layer) that takes the scalar covariates and the previous bearing, and produces parameters for a parametric movement kernel (a mixture of wrapped distributions centred on the current location).

The two log-probability surfaces are summed pixel-wise to give the joint predicted probability map for the animal's next step.

In [ ]:
model = deepssf.ConvJointModel(params).to(DEVICE)
n_params = sum(p.numel() for p in model.parameters())
print(f"Model parameters : {n_params:,}")

### Print the model structure

In [ ]:
print(model)

## `negativeLogLikeLoss` 
Scores each predicted probability map against the true next-step pixel index; minimising it pushes probability mass toward observed locations.

In [ ]:
loss_fn = deepssf.negativeLogLikeLoss(reduction="mean")

## `make_optimisers()` 
Creates two Adam optimisers with separate learning rates:

- `lr_habitat`
    * controls the CNN (habitat) head; typically higher
- `lr_movement`
    * controls the movement-kernel MLP; kept lower for stability

It also returns a `ReduceLROnPlateau` scheduler for each optimiser that reduces the learning rate by a factor of `scheduler_factor` (default = 0.1) if validation loss does not improve for `scheduler_patience` epochs (default = 5).


In [ ]:
optimisers, schedulers = deepssf.make_optimisers(
    model, 
    lr_habitat=1e-4, 
    lr_movement=1e-5, 
    scheduler_patience=3, 
    scheduler_factor=0.5
)

## `EarlyStopping` 
Monitors validation loss and saves the best model weights to disk whenever a new minimum is reached.  Training stops automatically if the loss does not improve for `patience` consecutive epochs, preventing overfitting.

- `patience`
    * If the loss doesn't improve for `patience` epochs, terminate training

In [ ]:
early_stop = deepssf.EarlyStopping(
    patience=10,
    verbose=True,
    monitor="both",                 # stop only when BOTH heads have plateaued
    path=str(OUTPUT_DIR / "best_model.pt"),
    checkpoint_on="active",         # a habitat-only stage saves on habitat loss
    # Also keep each head's own best epoch, across the whole run
    head_paths={
        "habitat":  str(OUTPUT_DIR / "best_model_habitat.pt"),
        "movement": str(OUTPUT_DIR / "best_model_movement.pt"),
    },
)

## `fit()` 
Runs the full training loop.

- Inputs: 
    * `model`
        - the name of the deepSSF model specified using `ConvJointModel` above
    * `image_trim_pixels` 
        - how many outer pixel rings to trim from the output images before saving to prevent artifacts with the colour scaling due to the image padding being pulled into the image during the convolution layers.
    * `dl_train`
        - train dataloader (contains the training data)
    * `dl_val`
        - validation dataloader (contains the validation data used to assess the learning rate and early stopping)
    * `loss_fn`
        - loss function
    * `optimisers`
        - optimisers used to update the model weights
    * `schedulers`
        - learning-rate schedulers (used to update the learning rates)
    * `early_stopping`
        - stores the validation loss and stops the model training if it doesn't improve for a given number of epochs (set using `patience` in the `EarlyStopping` definition)
    * `snapshot_dir`
        - where to save images of the training progress
    * `snapshot_item`
        - which sample (movement step) from the validation dataset to run the model on to generate the movement, habitat selection and next-step probability surfaces 

- Outputs: 
    * history dict with keys `train_losses` and `val_losses` (one scalar value per completed epoch)

**Index 0 of every history list is a pre-training baseline** — a validation pass run before the first epoch, with `train_losses[0] = nan` and `stage[0] = -1`. An untrained habitat surface is uniform over the window, and the gap between that and the final `val_habitat_losses` is a direct read on how much the habitat head actually learned.

### Staged training

`stages` runs the two sub-networks as a coordinate ascent on the same joint likelihood, rather than reweighting it:

1. **movement only** — get a sensible availability kernel in place first, since habitat is fitted *conditional* on it
2. **habitat only** — habitat now runs to its own convergence, on its own early-stopping clock, with the movement surface held fixed in the likelihood as an offset
3. **both** — a joint pass to settle

A frozen sub-network keeps contributing its surface to the loss; only its weights stop moving. This is why staging is used instead of `negativeLogLikeLoss(freeze_movement=True)` — that flag *drops* the movement surface from the combined loss, so habitat would have to explain the movement structure too and would learn a distance-decay bump centred on the previous location, which is the opposite of a clean resource-selection surface.

Early stopping ends the current *stage* and advances to the next, so the final stage's early stop ends the run.

Every epoch, a plot of the training and validation loss, and the predicted probability maps (movement, habitat selection and next-step) are written to `SNAPSHOT_DIR`, which will be turned into a GIF in the next cell. 


In [ ]:
history = deepssf.fit(
    model=model,
    image_trim_pixels=4,
    window_size=WINDOW_SIZE,
    dl_train=dl_train,
    dl_val=dl_val,
    loss_fn=loss_fn,
    optimisers=optimisers,
    schedulers=schedulers,
    stages=[
        {"epochs": 20, "train": ("movement",)},   # availability kernel first
        {"epochs": 50, "train": ("habitat",)},    # habitat to its own convergence
        {"epochs": 30, "train": ("habitat", "movement")},  # joint settle
    ],
    early_stopping=early_stop,
    snapshot_dir=str(SNAPSHOT_DIR),
    snapshot_item=250,
    batch_size=BATCH_SIZE,
)

## `create_gif()`
Assembles the snapshots in `snapshot_dir` into an animation so you can see how
the predictions evolve as training progresses.

Display it with `IPython.display.Image`, which reads the file and embeds the
bytes in the notebook's output.  A hand-written `<img src="outputs/....gif">`
tag does *not* work here: the notebook's HTML output is rendered in a sandboxed
webview whose base URL is not the notebook's directory, so a relative path
resolves to nothing and you get an empty box.  Nothing is generating a still
frame — there is simply no image being loaded at all.  Embedding also means the
animation survives being exported to HTML or shared as a standalone `.ipynb`.

In [ ]:
gif_path = str(OUTPUT_DIR / "training_progress.gif")
deepssf.create_gif(str(SNAPSHOT_DIR), gif_path, fps=5)
print(f"Training GIF → {gif_path}")

# Image() embeds the file's bytes in the output, so it renders wherever the
# notebook is opened.  Leave it as the last expression to display it.
Image(filename=gif_path)

## Training - loss across epochs

How did the predictive performance on the validation dataset change across the training period?

In [ ]:
# Three panels rather than one: the habitat and movement components move on very
# different scales, so sharing an axis would flatten the habitat curve into a
# horizontal line.  x = 0 is the pre-training baseline recorded by fit().
uniform_nll = np.log(WINDOW_SIZE ** 2)
epochs = np.arange(len(history["val_losses"]))
stages = np.asarray(history["stage"])

fig, axes = plt.subplots(1, 3, figsize=(14, 3.6), constrained_layout=True)

panels = [
    ("Combined", history["val_losses"], history["train_losses"]),
    ("Habitat", history["val_habitat_losses"], None),
    ("Movement", history["val_movement_losses"], None),
]

for ax, (title, val, train) in zip(axes, panels):
    if train is not None:
        ax.plot(epochs, train, color="0.6", lw=1.5, label="train")
    ax.plot(epochs, val, color="C0", lw=1.5, label="val")
    if title == "Habitat":
        # A uniform surface over the window scores log(H*W), so this line is the
        # "learned nothing" mark — the whole question for the habitat head is how
        # far below it the curve gets.  Movement sits far below it from the first
        # epoch, and drawing it there would only compress that panel's axis.
        ax.axhline(uniform_nll, color="0.4", ls="--", lw=1,
                   label=f"uniform ({uniform_nll:.2f})")
    # Shade the staged-training blocks so the schedule is readable off the curve.
    for s in np.unique(stages[stages >= 0])[1::2]:
        within = epochs[stages == s]
        ax.axvspan(within.min() - 0.5, within.max() + 0.5, color="0.92", zorder=0)
    ax.set_xlabel("Epoch (0 = untrained baseline)")
    ax.set_ylabel("NLL (nats)")
    ax.set_title(title)
    ax.legend(frameon=False, fontsize=8)
    ax.spines[["top", "right"]].set_visible(False)

fig.suptitle("Training history — combined loss and its two components")
plt.savefig(str(OUTPUT_DIR / "loss_history.png"), dpi=100)
plt.show()


## Choosing which checkpoint to use
`fit()` leaves the model holding its **final-epoch** weights, not the best ones,
so load a checkpoint before testing or simulating.

- `best_model.pt` — the model to use for **simulation**.  Both heads from one
  epoch, so the joint likelihood is the one that was actually measured.
- `best_model_habitat.pt` / `best_model_movement.pt` — each head at its own best
  epoch, tracked across the whole run.

For simulating trajectories, you will likely want to use the best performing model overall (`best_model.pt`).

In [ ]:
# The joint model — what the simulations further down should use
ckpt = deepssf.load_checkpoint(str(OUTPUT_DIR / "best_model.pt"), model)
print(f"best_model.pt         saved on '{ckpt['checkpoint_criterion']}' "
      f"= {ckpt['checkpoint_score']:.6f}")

hab_ckpt = deepssf.load_checkpoint(str(OUTPUT_DIR / "best_model_habitat.pt"))
print(f"best_model_habitat.pt  saved on '{hab_ckpt['checkpoint_criterion']}' "
      f"= {hab_ckpt['checkpoint_score']:.6f}")


# Test the model against withheld data

## Convert the NumPy rasters loaded earlier into PyTorch tensors.
Both validate_next_step_probs() and simulate_trajectory() call `get_landscape(month_index)` at each step to fetch the current environmental layers as a list of 2-D tensors.

With static (non-seasonal) rasters the month_index argument is unused. For seasonal data (e.g. monthly NDVI composites) you would index into a stack of layers here based on the current month.

In [ ]:
# validate_next_step_probs() and simulate_trajectory() both stack the list returned
# by get_landscape() along a new channel dimension, so every element must be a *2-D*
# tensor.  Unbind the band dimension of any multi-band layer so the channel order
# matches what MovementDataset fed the model during training.
landscape_tensors = []
for arr in env_layers.values():
    t = torch.from_numpy(np.asarray(arr).astype(np.float32))
    landscape_tensors.extend([t] if t.ndim == 2 else list(t))

print(f"Landscape channels: {len(landscape_tensors)}  (each {tuple(landscape_tensors[0].shape)})")

def get_landscape(_month_index):
    # Returns one 2-D tensor per spatial channel, in the same order as LAYER_PATHS.
    return landscape_tensors

# Take the last 10 % of steps as a test sample.  CAVEAT: make_dataloaders() uses a
# random_split, so these steps were mostly *seen during training* — the numbers below
# are not a clean out-of-sample estimate.  Because step_df is ordered by individual
# name, this slice is also just the last few pigs alphabetically rather than the
# latest steps in time.  Hold out individuals or a date range explicitly for a real test.
n_test      = int(len(step_df) * 0.1)
test_sample = step_df.iloc[-n_test:].reset_index(drop=True)
print(f"Testing on {len(test_sample):,} steps")

## `validate_next_step_probs()`
Runs the model forward on each test step and records the probability mass assigned to the *true* next location.

- **Inputs**: 
    * `model`
    * `test_sample`
        - test dataframe that contains movement steps (from the data loaded above)
     * `get_landscape`
        - retrieves the landscape layers (spatial covariates) that were used to train the model
    * `transform`
        - raster Affine transform
    * `window_size`
        - window size of the local availability window (should be the same as for training)
    * `scalar_cols`
        - scalar column names (should be set from above)
    * `yday_col`
        - which column has the yday column to ensure the correct spatial layers (if dynamic through time) are indexed
    * `month_index_fn`
        - a function mapping day-of-year → month index (used by the `get_landscape` function). Here we are using static spatial layers so we can set it to 0.

- **Outputs**: 
    * `test_sample` dataframe with three new columns appended:
        - habitat_prob   — probability at the true location from the habitat head alone
        - move_prob      — probability at the true location from the movement head alone
        - next_step_prob — joint (habitat × movement) probability at the true location

Higher values mean the model performed better (there was higher predicted probability at the observed next-step). 


In [ ]:
# Use the last 10 % of steps as the test set (matches the DataLoader split)
n_test      = int(len(step_df) * 0.1)
test_sample = step_df.iloc[-n_test:].reset_index(drop=True)
print(f"Validating on {len(test_sample):,} steps")

val_results = deepssf.validate_next_step_probs(
    model,
    test_sample,
    get_landscape=get_landscape,
    transform=raster_transform,
    window_size=WINDOW_SIZE,
    scalar_cols=tuple(SCALAR_COLS),
    yday_col="yday_t1",
    bearing_col="bearing_tm1",
    month_index_fn=lambda _yday: 0,   # static layers
)
print(val_results[["habitat_prob", "move_prob", "next_step_prob"]].describe())

## Plot the predictive performance on the held-out data

In [ ]:
# The uniform probability of any cell in the window is 1/(W*H) = 1/(W^2) since W=H.
uniform_prob = 1/(WINDOW_SIZE ** 2)
uniform_prob

fig, axes = plt.subplots(1, 3, figsize=(12, 3))
for ax, col, title in zip(
    axes,
    ["habitat_prob", "move_prob", "next_step_prob"],
    ["Habitat", "Movement", "Next-step"],
    strict=True,
):
    data = val_results[col].dropna()
    ax.hist(data, bins=40, edgecolor="white")
    ax.axvline(
        uniform_prob,
        color="black",
        linestyle="--",
        linewidth=1.5,
        label="Uniform (1/W²)",
    )
    ax.set_title(title)
    ax.set_xlabel("Probability")
    ax.ticklabel_format(axis="x", style="sci", scilimits=(-3, 3))
    ax.legend()

axes[0].set_ylabel("Count")
plt.suptitle("Next-step probability distributions (test set)", y=1.02)
plt.tight_layout()
plt.savefig(str(OUTPUT_DIR / "validation_probs.png"), dpi=100, bbox_inches="tight")
plt.show()

# Habitat selection across the full landscape

## `make_simulation_inputs`
This function creates the sin/cos encodings for specific days and hours of the year, and is used when simulating to convert from days of the year and hours into sin/cos encodings. It will return a tuple of length `n_steps`. 

Here, we just want the sin/cos encodings for a single point in time, so we just set `n_steps` to 1 and input a day of the year (`PRED_YDAY`, integer) and fractional hour (`PRED_HOUR`, float).

## `predict_habitat_landscape()`
Everything so far has run the model on a `WINDOW_SIZE` crop centred on the
animal.  But the habitat sub-network is *fully convolutional* — multiple layers of 3x3
convolutions with stride 1 and padding 1, no pooling and no dense layer — so the
same trained filters can be swept across the entire raster in a single pass.
Convolution is translation-equivariant, so the value at a pixel is exactly what
the model would have produced for a window centred on that pixel.

That gives a landscape-scale habitat selection surface.  Read it as a **relative
ranking of cells**: "if the animal could access anywhere on this landscape, how
attractive is each cell at a certain point in time". It ignores the movement process, which defines availability. The two are only combined over a local window, which is what `simulate_next_step()` does.

- **Inputs**:
    * `model`
        - the trained model; only `model.conv_habitat` is used
    * `landscape_rasters`
        - the spatial covariates as a list of 2-D tensors, in the same order and
          on the same scaling as training — i.e. exactly what `get_landscape()`
          returns
    * `scalars`
        - the scalar covariates for the moment being predicted, in `SCALAR_COLS`
          order.  `make_simulation_inputs()` builds this row for us, so the
          encoding cannot drift from what the simulation uses.  Each value is
          broadcast to a constant layer covering the whole landscape (the same
          thing `Scalar_to_Grid_Block` does for a window — that block can't be
          reused here because its output size is fixed to `WINDOW_SIZE`)
    * `normalise`
        - subtract the log-sum-exp over the whole landscape, making the surface
          a log-probability that sums to 1 across all valid cells
    * `edge_buffer`
        - how many border pixels to mask as `NaN`.  Defaults to
          `habitat_edge_buffer(model)` — the exact width the zero-padding
          reaches, 1 pixel per conv layer, so 4 here.  Those pixels saw padding
          rather than landscape and are not comparable with the interior
    * `chunk_rows`
        - process this many raster rows at a time.  A full-landscape activation
          is several hundred MB, enough to exhaust a GPU; the chunks overlap by
          `edge_buffer` so the result is identical to a single pass

- **Outputs**: a `[height, width]` NumPy array of log-probabilities, `NaN` in the
  border.  Take `np.exp()` for probabilities.

In [ ]:
# Predict for one moment in time: 9:30 am on day 200.  
PRED_YDAY = 200 # what day of the year to simulate (1–365, integer)
PRED_HOUR = 9.5 # what hour of the day to simulate (0–24, float)

# Get one the sin/cos encodings at PRED_YDAY and PRED_HOUR
scalars_pred = deepssf.make_simulation_inputs(
    n_steps=1, 
    starting_yday=PRED_YDAY, 
    starting_hour=PRED_HOUR,
    time_between_steps=1.0,  # this isn't used here, but this argument is used to create the step intervals (in hours)
)[0][0]
print(f"scalar covariates ({', '.join(SCALAR_COLS)}):\n  {scalars_pred}")

habitat_log = deepssf.predict_habitat_landscape(
    model,
    landscape_rasters=get_landscape(0),   # static layers → month index 0
    scalars=scalars_pred,
)
print(f"\nSurface {habitat_log.shape}, masked border "
      f"{deepssf.habitat_edge_buffer(model)} px")
print(f"log-probability range: {np.nanmin(habitat_log):.2f} to "
      f"{np.nanmax(habitat_log):.2f}")
print(f"probabilities sum to  : {np.nansum(np.exp(habitat_log)):.4f}")

### Plot the landscape-scale habitat selection surface
Plotted on the **log** scale.  Across millions of cells the probabilities are
tiny (a uniform surface would give something on the order of ~1e-7 per cell) and a handful of very
attractive cells dominate the linear range, so a linear colour scale washes the
whole map out.  The colour limits are clipped to percentiles for the same
reason.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))

# Percentile limits for the colour scale: otherwise a few extreme cells can dominate the colour range
vmin, vmax = np.nanpercentile(habitat_log, [1.0, 99.0])

# Convert the fractional hour to hours and minutes for the title
h, m = divmod(round(PRED_HOUR * 60), 60)

rasterio.plot.show(
    habitat_log, transform=raster_transform, ax=axes[0],
    cmap="viridis", vmin=vmin, vmax=vmax,
)
axes[0].set_title(f"Habitat selection (log prob) — yday {PRED_YDAY}, "
                  f"{h}:{m:02d}")

rasterio.plot.show(
    habitat_log, transform=raster_transform, ax=axes[1],
    cmap="viridis", vmin=vmin, vmax=vmax,
)
axes[1].plot(
    step_df["x1_"], step_df["y1_"],
    "r.", alpha=0.25, markersize=0.6, label="observed GPS fixes",
)
axes[1].legend(loc="upper right", markerscale=10)
axes[1].set_title("with observed locations overlaid")

for ax in axes:
    ax.set_xlabel("Easting (m, projected)")
    ax.set_ylabel("Northing (m, projected)")

# One colourbar for both panels
fig.colorbar(axes[0].images[0], ax=axes, shrink=0.8, label="log probability")
plt.savefig(str(OUTPUT_DIR / "habitat_landscape.png"), dpi=120, bbox_inches="tight")
plt.show()

### How does habitat selection change through the day?
The scalar covariates are inputs to the habitat CNN, so the *same* landscape
produces a different surface at different times of day. Re-running the prediction
at different times of day shows what the model has learned.

Each surface is normalised over the landscape independently, so the panels show
where selection is concentrated at that hour, not the overall level of activity.

These surfaces **only** show the habitat selection, and not where would expect to
find pigs, as these maps do not take movement into consideration. To generate 
distribution maps we need to simulate movement trajectories. 

In [ ]:
PRED_HOURS = [3, 9.5, 15, 21]

fig, axes = plt.subplots(2, 2, figsize=(13, 10))
surfaces = {}
for ax, hour in zip(axes.flat, PRED_HOURS, strict=True):
    scalars_h = deepssf.make_simulation_inputs(
        n_steps=1, starting_yday=PRED_YDAY, starting_hour=float(hour)
    )[0][0]
    surf = deepssf.predict_habitat_landscape(
        model, landscape_rasters=get_landscape(0), scalars=scalars_h
    )
    surfaces[hour] = surf

    # Convert the fractional hour to hours and minutes for the title
    h, m = divmod(round(hour * 60), 60)

    lo, hi = np.nanpercentile(surf, [1.0, 99.0])
    rasterio.plot.show(
        surf, transform=raster_transform, ax=ax, cmap="viridis", vmin=lo, vmax=hi
    )
    ax.set_title(f"{h:02d}:{m:02d}")
    ax.set_xlabel("Easting (m)")
    ax.set_ylabel("Northing (m)")

plt.suptitle(f"Habitat selection through the day (yday {PRED_YDAY})")
plt.tight_layout()
plt.savefig(str(OUTPUT_DIR / "habitat_landscape_by_hour.png"), dpi=120)
plt.show()

# How much does the surface actually move between hours?  If these differences
# are ~0 the model has not learned any diel variation in habitat use.
ref = surfaces[PRED_HOURS[0]]
for hour in PRED_HOURS[1:]:
    h, m = divmod(round(hour * 60), 60)
    diff = np.nanmax(np.abs(surfaces[hour] - ref))
    print(f"max |log-prob difference| between 00:00 and {h:02d}:{m:02d}: {diff:.3f}")

## Create a GIF of the hourly habitat selection surfaces

Save each hourly habitat selection map as a PNG (within a subdirectory called 'habitat_selection_maps'), and then combine them into a GIF. 

These maps are just the habitat selection probabilities from the convolution layers, and ignore the movement process. They are therefore more of a diagnostic tool to assess how habitat selection changes throughout the day.  

In [ ]:
HABITAT_MAP_DIR = OUTPUT_DIR / "habitat_selection_maps"
HABITAT_MAP_DIR.mkdir(exist_ok=True)

# Predict all 24 hours first, then draw them.  The two passes are what lets every
# frame share one colour scale: with per-frame percentile limits (as in the 2x2
# figure above) each surface is re-normalised to its own range, so the animation
# flickers without showing where selection has actually moved.  
hourly_surfaces = {}
for hour in range(24):
    scalars_h = deepssf.make_simulation_inputs(
        n_steps=1, 
        starting_yday=PRED_YDAY, 
        starting_hour=float(hour),
        time_between_steps=1.0,
    )[0][0]
    h, m = divmod(round(hour * 60), 60)
    hourly_surfaces[hour] = deepssf.predict_habitat_landscape(
        model, landscape_rasters=get_landscape(0), scalars=scalars_h
    )
    print(f"  {h:02d}:{m:02d} done", end="\r")

# Shared colour limits over the whole day, clipped to the same percentiles used
# for the single-hour map (a handful of extreme cells otherwise take the range).
VMIN = min(np.nanpercentile(s, 1.0) for s in hourly_surfaces.values())
VMAX = max(np.nanpercentile(s, 99.0) for s in hourly_surfaces.values())
print(f"shared colour limits: {VMIN:.2f} to {VMAX:.2f}")

for hour, surf in hourly_surfaces.items():
    fig, ax = plt.subplots(figsize=(7.5, 6))
    rasterio.plot.show(
        surf, transform=raster_transform, ax=ax,
        cmap="viridis", vmin=VMIN, vmax=VMAX,
    )
    h, m = divmod(round(hour * 60), 60)
    ax.set_title(f"Habitat selection (log prob) — yday {PRED_YDAY}, {h:02d}:{m:02d}")
    ax.set_xlabel("Easting (m, projected)")
    ax.set_ylabel("Northing (m, projected)")
    fig.colorbar(ax.images[0], ax=ax, shrink=0.8, label="log probability")

    # Zero-padded hour so the filenames sort into clock order.  Note the absence
    # of bbox_inches="tight": it trims each figure to its own content, and frames
    # of differing pixel size cannot be stacked into a GIF.
    fig.savefig(HABITAT_MAP_DIR / f"habitat_hour_{hour:02d}.png", dpi=300)
    plt.close(fig)   # close, or 24 figures are drawn into the notebook output

print(f"Wrote 24 frames to {HABITAT_MAP_DIR}")

### Display the GIF of hourly habitat selection

In [ ]:
import imageio.v2 as imageio

habitat_gif = OUTPUT_DIR / "habitat_selection_hourly.gif"

# Read the frames back off disk, sorted by filename — the zero-padded hour in the
# name puts them in clock order.  create_gif() (used for the training snapshots
# above) is deliberately not used here: it orders frames by an `epoch_<n>` field
# in the filename and, with no such field to key on, would leave them in whatever
# order the filesystem happened to list them.
frame_paths = sorted(HABITAT_MAP_DIR.glob("habitat_hour_*.png"))
frames = [imageio.imread(p) for p in frame_paths]

# duration is milliseconds *per frame*: half a second an hour 
# (`fps=` is accepted but deprecated in imageio v2 and warns.)
# loop=0 means loop forever rather than play once.
imageio.mimsave(habitat_gif, frames, duration=200, loop=0)
print(f"{len(frames)} frames → {habitat_gif} "
      f"({habitat_gif.stat().st_size / 1e6:.1f} MB)")

# As with the training GIF, Image() embeds the bytes so it renders wherever the
# notebook is opened.
Image(filename=str(habitat_gif))

### Save the hourly habitat selection surfaces as a raster (TIF)
Writing them out with the raster's transform and CRS means they can be opened
directly in QGIS/ArcGIS or read back with `rasterio`, and overlaid on the other
layers in `mapping/`.

One **band per hour** rather than one file per hour: the 24 surfaces share a
grid, a transform and a CRS, so a single stack keeps them together and lets GIS
software step through the day with the band slider.  Band *n* is hour *n - 1*
(GDAL numbers bands from 1), and each band carries its hour in both its
description and its tags — so `src.read(15)` is 14:00.


In [ ]:
# The CRS is not returned by load_environmental_layers(), so read it off the
# source raster the transform came from.
with rasterio.open(next(iter(LAYER_PATHS.values()))) as src:
    raster_crs = src.crs

# Reuses the surfaces predicted for the GIF above, so run that cell first.
# Sorting the keys rather than trusting dict insertion order means band n is
# always hour n - 1, however the surfaces were generated.
hours_sorted = sorted(hourly_surfaces)
height, width = hourly_surfaces[hours_sorted[0]].shape

habitat_tif = OUTPUT_DIR / f"habitat_landscape_yday{PRED_YDAY}_hourly.tif"
with rasterio.open(
    habitat_tif, "w",
    driver="GTiff",
    height=height,
    width=width,
    count=len(hours_sorted),      # one band per hour
    dtype="float32",
    crs=raster_crs,
    transform=raster_transform,
    nodata=np.nan,          # the masked edge buffer
    compress="deflate",
) as dst:
    for band, hour in enumerate(hours_sorted, start=1):   # GDAL bands are 1-indexed
        dst.write(hourly_surfaces[hour].astype("float32"), band)
        # The description is what QGIS shows in its band dropdown; the tags keep
        # the hour machine-readable for anything parsing the metadata.
        dst.set_band_description(band, f"habitat selection log prob {hour:02d}:00")
        dst.update_tags(band, hour=hour, yday=PRED_YDAY)
    dst.update_tags(prediction_yday=PRED_YDAY, hours=",".join(str(h) for h in hours_sorted))

print(f"Wrote {habitat_tif}")
print(f"  {len(hours_sorted)} bands, {width} x {height} px "
      f"({habitat_tif.stat().st_size / 1e6:.1f} MB)")

# Simulate new data

Select starting parameters for the model.

We've set the starting location to be the first location of the observed data.

In [ ]:
# the starting id of the step (should be 1)
START_IDX = 1

# where should the simulation start?  
# here we use the first step in the observed data, but any valid (x, y) within the
# raster extent could be used. 
start_x = step_df["x1_"].iloc[START_IDX]
start_y = step_df["y1_"].iloc[START_IDX]
print(f"Starting location  : ({start_x:.0f}, {start_y:.0f})")

# the timestamp of the first step in the observed data
START_TIME = step_df["t1_"].iloc[START_IDX]
# extract the day of year and hour of day as floats for the simulation inputs
START_YDAY = START_TIME.timetuple().tm_yday
START_HOUR = START_TIME.hour + START_TIME.minute / 60.0

print(f"Starting timestamp : {START_TIME}")
print(f"Starting time      : day {START_YDAY:.0f}, {START_HOUR:.2f} h")

# hours between simulated steps - this should ideally be the same as the observed step interval, 
# but the model can be used to simulate at any interval
# (but it may not have learned a meaningful movement kernel for other intervals).
STEP_INTERVAL = 1.0 

# Simulate a trajectory

## `simulate_trajectory()` 
Generates a sequence of steps by iteratively:
1. Extracting a spatial window centred on the current location.
2. Running the model to obtain a joint log-probability surface over that window.
3. Sampling the next pixel from that surface (softmax-normalised probabilities).
4. Converting the sampled pixel back to projected coordinates and advancing time.

- **Inputs**: 
    * `model`
        - the trained model
    * `get_landscape`
        - retrieves the landscape layers (spatial covariates) that were used to train the model
    * `transform`
        - raster Affine transform
    * `start_x`
        - x coordinate of starting location
    * `start_y`
        - y coordinate of starting location
    * `n_steps`
        - number of steps
    * `starting_yday`
        - starting day-of-year 
    * `starting_hour`
        - starting hour-of-day
    * `start_time`
        - what time should the simulation start
    * `time_between_steps`
        - time between steps in hours
    * `window_size`
        - size in pixels of the local landscape
    * `month_index_fn`
        - a function mapping day-of-year → month index (used by the `get_landscape` function). Here we are using static spatial layers so we can set it to 0.

- **Outputs**: DataFrame with one row per simulated step containing:
    * `x`, `y`           
        — projected coordinates of the simulated location
    * `hour`, `yday`     
        — time variables updated at each step
    * `month_index`
        - the month index used to index the spatial variables (will be 0 if using static spatial covariates)

In [ ]:
sim_df = deepssf.simulate_trajectory(
    model,
    get_landscape=get_landscape,
    transform=raster_transform,
    start_x=start_x,
    start_y=start_y,
    n_steps=1000,
    starting_yday=START_YDAY,
    starting_hour=START_HOUR,
    start_time=START_TIME,
    time_between_steps=STEP_INTERVAL,
    window_size=WINDOW_SIZE,
    month_index_fn=lambda _yday: 0,   # static layers
)
print(f"Simulated {len(sim_df)} steps")
sim_df.head()

### Plot the simulated data

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))
rasterio.plot.show(
    env_layers["ndvi"], transform=raster_transform, ax=ax, cmap="viridis"
)
ax.plot(
    step_df["x1_"], step_df["y1_"],
    "w-o", alpha=0.5, markersize=1, linewidth=0.8, label="observed",
)
ax.plot(
    sim_df["x"], sim_df["y"],
    "r-o", markersize=1, linewidth=0.8, label="simulated",
)
ax.set_xlabel("Easting (m, projected)")
ax.set_ylabel("Northing (m, projected)")
ax.legend()
ax.set_title("Simulated vs observed buffalo locations")
plt.tight_layout()
plt.savefig(str(OUTPUT_DIR / "simulated_trajectory.png"), dpi=100)
plt.show()

# Plot with an interactive map

## `plot_trajectories_folium()`
A static figure is fine for checking the overall shape of a trajectory, but not for
asking *where* the animal actually went.  This draws the simulated (and optionally
observed) locations over a satellite basemap so you can zoom in on individual steps.

- **Inputs**:
    * `trajectories`
        - a simulated-locations DataFrame.  If it has a `trajectory_id` column
          (as `simulate_trajectories()` returns) each id is drawn as its own
          toggleable line; a plain `sim_df` is drawn as a single line.
    * `observed`
        - observed locations to overlay, e.g. `step_df`.  Drawn as points rather
          than a line by default, because `step_df` holds 28 individuals and a
          single line would connect them with jumps that never happened.
    * `src_crs`
        - CRS of the input coordinates.  Everything is reprojected to EPSG:4326
          (lat/lon) for the web map — our data is in EPSG:3112.
    * `basemap`
        - `"esri_imagery"` (default), `"esri_topo"`, `"openstreetmap"`, or any
          XYZ tile-URL template.  See the folium docs for more:
          <https://python-visualization.github.io/folium/latest/user_guide/raster_layers/tiles.html>
    * `max_observed_points`
        - observed locations are thinned to this many points.  folium renders
          every point as its own element in the page, so all 100k+ fixes would
          lock up the notebook.
    * `save_path`
        - optionally also write a standalone HTML file.

- **Outputs**: a `folium.Map` — leave it as the last line of a cell to display it.

`folium` is an optional dependency: `pip install folium` (or
`pip install "deepSSF[maps]"`).

In [ ]:
fmap = deepssf.plot_trajectories_folium(
    sim_df,
    observed=step_df,
    src_crs="EPSG:3112",          # projected CRS of the GPS data and rasters
    basemap="esri_imagery",
    zoom_start=12,
    max_observed_points = 10000, 
    save_path=str(OUTPUT_DIR / "simulated_trajectory_map.html"),
)
fmap

# Save the trajectory to a CSV

## `save_trajectories()`
Writes the simulated locations to `outputs/deepSSF_trajectories/`.  The filename
records how many trajectories, how many steps and the date, and the function
**never overwrites an existing file** — re-running the cell appends a `_2`, `_3`,
... run counter instead, so repeated runs accumulate rather than clobber.

- **Inputs**:
    * `trajectories` — a DataFrame (or list of DataFrames)
    * `output_dir` — created if it doesn't exist
    * `prefix` — leading part of the filename
    * `split` — write one file per `trajectory_id` instead of one combined file
- **Outputs**: the list of paths written


In [ ]:
# set up a directory to hold the CSV files for the simulated trajectories
TRAJ_DIR = OUTPUT_DIR / "deepSSF_trajectories"

# save the simulated trajectory to a CSV file
deepssf.save_trajectories(sim_df, TRAJ_DIR, prefix="deepSSF_buffalo")

# Simulating many trajectories

## `simulate_trajectories()`
Runs `n_trajectories` simulations and returns them stacked in one long-format
DataFrame with `trajectory_id` and `step` columns.

Three `method`s are available:

| `method` | how it works | when to use it |
| --- | --- | --- |
| `"batched"` *(default)* | every trajectory takes its step together, so the model sees a batch of `n_trajectories` windows per forward pass instead of one | almost always — the forward pass is the whole cost of the simulation, and it is far better used at batch size 50 than at batch size 1 |
| `"sequential"` | a plain loop over `simulate_trajectory()` | reference behaviour; needed if trajectories start at *different times* |
| `"parallel"` | the same loop across a thread pool | when start times vary and you want some overlap; the speed-up is modest (PyTorch releases the GIL inside its ops, but torch is already using several threads per op) and results are not reproducible from a seed |

The one restriction on `"batched"` is that all trajectories share a clock, so
`starting_yday` / `starting_hour` must be scalars.  Start *locations* can still
differ per trajectory — pass a list to `start_x` / `start_y` to release simulated
animals from different points.

In [ ]:
N_TRAJECTORIES  = 1000   # how many simulated trajectories
N_SIM_STEPS     = 1000  # how many steps per trajectory

sim_many = deepssf.simulate_trajectories(
    model,
    get_landscape=get_landscape,
    transform=raster_transform,
    start_x=start_x,
    start_y=start_y,
    n_steps=N_SIM_STEPS,
    n_trajectories=N_TRAJECTORIES,
    method="batched",
    starting_yday=START_YDAY,
    starting_hour=START_HOUR,
    start_time=START_TIME,
    time_between_steps=1.0,
    window_size=WINDOW_SIZE,
    month_index_fn=lambda _yday: 0,   # static layers
)
print(f"{sim_many['trajectory_id'].nunique()} trajectories x "
      f"{N_SIM_STEPS} steps = {len(sim_many):,} locations")
sim_many.head()

### Releasing simulated animals from different starting points
Passing a sequence to `start_x` / `start_y` gives each simulated animal its own
release site.  Here we sample starting locations from the observed data.

In [ ]:
# Draw starting locations from the observed fixes
start_locations = step_df.sample(N_TRAJECTORIES, random_state=42)
start_locations

### Simulate the trajectories

In [ ]:
sim_scattered = deepssf.simulate_trajectories(
    model,
    get_landscape=get_landscape,
    transform=raster_transform,
    start_x=start_locations["x1_"].to_numpy(),
    start_y=start_locations["y1_"].to_numpy(),
    n_steps=N_SIM_STEPS,
    n_trajectories=N_TRAJECTORIES,
    method="batched",
    starting_yday=START_YDAY,
    starting_hour=START_HOUR,
    start_time=START_TIME,
    window_size=WINDOW_SIZE,
    month_index_fn=lambda _yday: 0,
)
print(f"{len(sim_scattered):,} locations from {N_TRAJECTORIES} release sites")

### Save the simulated trajectoes
`split=False` (the default) writes one combined CSV with the `trajectory_id`
column; `split=True` writes one file per simulated individual.


In [ ]:
deepssf.save_trajectories(sim_many, TRAJ_DIR, prefix="deepSSF_buffalo_many")
deepssf.save_trajectories(sim_scattered, TRAJ_DIR, prefix="deepSSF_buffalo_scattered")

# ...or one file per simulated animal:
# save_trajectories(sim_many, TRAJ_DIR, prefix="deepSSF_pigs", split=True)

### To read in previously saved trajectories

In [ ]:
# # to read previously run simulations from disk, rather than re-running the simulation above
# SIM_PATH = TRAJ_DIR / "deepSSF_buffalo_scattered_1000traj_1000steps_2026-10-22.csv"
# sim_scattered = pd.read_csv(SIM_PATH)
# print(f"Read {len(sim_scattered):,} locations from {SIM_PATH}")
# sim_scattered.head()

### Plot all of the simulated trajectories


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 6))
for ax, df, title in zip(
    axes,
    [sim_many, sim_scattered],
    ["Same release site", "Different release sites"],
    strict=True,
):
    rasterio.plot.show(
        # select the first channel (NDVI) for the background, use the index to change layers
        landscape_tensors[0].numpy(), transform=raster_transform, ax=ax, cmap="viridis"
    )
    ax.plot(
        step_df["x1_"], step_df["y1_"],
        "w.", alpha=0.2, markersize=0.5, label="observed",
    )
    # One colour per simulated individual
    for tid, sub in df.groupby("trajectory_id"):
        ax.plot(sub["x"], sub["y"], "-", linewidth=0.7, alpha=0.9, label=f"sim {tid}")
    ax.set_xlabel("Easting (m, projected)")
    ax.set_ylabel("Northing (m, projected)")
    ax.set_title(title)

plt.suptitle(f"{N_TRAJECTORIES} simulated trajectories")
plt.tight_layout()
plt.savefig(str(OUTPUT_DIR / "simulated_trajectories.png"), dpi=100)
plt.show()

### A subset of the same trajectories on the interactive map
Each trajectory gets its own colour and its own entry in the layer control (top
right), so individual animals can be switched on and off.

In [ ]:
# folium writes every trajectory into the page as its own feature group, so all
# 599 make a map too heavy to open.  Draw a subset — the heatmap below is what
# summarises the full set.
N_MAP_TRAJECTORIES = 10
map_ids = sim_scattered["trajectory_id"].unique()[:N_MAP_TRAJECTORIES]

fmap_traj = deepssf.plot_trajectories_folium(
    sim_scattered[sim_scattered["trajectory_id"].isin(map_ids)],
    observed=step_df,
    src_crs="EPSG:3112",
    basemap="esri_imagery",
    zoom_start=11,
    max_observed_points = 100000,
)

traj_html = OUTPUT_DIR / "simulated_trajectories_map.html"
fmap_traj.save(str(traj_html))
print(f"Map saved: {traj_html}")
fmap_traj

## Visualise some data summaries of the simulated trajectories

Here we can check the summaries of the observed data against summaries of the simulated trajectories,
to see whether we've captured those behaviours.

### Calculate step lengths and turning angles from the simulated data

In [ ]:
sim_scattered

In [ ]:
sim_scattered_df = deepssf.prepare_movement_df(
    sim_scattered, 
    id_col="trajectory_id", 
    time_col="time", 
    x_col="x", 
    y_col="y"
)

n_dropped = len(sim_scattered) - len(sim_scattered_df)
print(f"Simulated movement steps : {len(sim_scattered_df):,}  ({n_dropped} dropped — last per individual)")
sim_scattered_df.head()

### Distribution of step lengths

In [ ]:
# Distribution of step lengths, with the median and mean marked
fig, ax = plt.subplots(figsize=(8, 4))
# observed step lengths
ax.hist(step_df["sl_"], bins=250, density=True, color="tab:blue", alpha=0.5, label="observed")
ax.axvline(step_df["sl_"].median(), color="tab:blue", lw=2, label="Median")
ax.axvline(step_df["sl_"].mean(), color="tab:blue", lw=2, ls="--", label="Mean")
# simulated step lengths
ax.hist(sim_scattered_df["sl_"], bins=250, density=True, color="tab:purple", alpha=0.5, label="simulated")
ax.axvline(sim_scattered_df["sl_"].median(), color="tab:purple", lw=2, label="Median")
ax.axvline(sim_scattered_df["sl_"].mean(), color="tab:purple", lw=2, ls="--", label="Mean")
ax.set_xlabel("Step length (m)")
ax.set_ylabel("Count")
ax.set_xlim(0, 2000)
ax.legend()
plt.tight_layout()
plt.show()

### Log of step lengths

In [ ]:
# Distribution of log step lengths, with the median and mean marked
fig, ax = plt.subplots(figsize=(8, 4))
# observed step lengths
ax.hist(np.log(step_df["sl_"]), bins=250, density=True, color="tab:blue", alpha=0.5, label="observed")
ax.axvline(np.log(step_df["sl_"].median()), color="tab:blue", lw=2, label="Median")
ax.axvline(np.log(step_df["sl_"].mean()), color="tab:blue", lw=2, ls="--", label="Mean")
# simulated step lengths
ax.hist(np.log(sim_scattered_df["sl_"]), bins=250, density=True, color="tab:purple", alpha=0.5, label="simulated")
ax.axvline(np.log(sim_scattered_df["sl_"].median()), color="tab:purple", lw=2, label="Median")
ax.axvline(np.log(sim_scattered_df["sl_"].mean()), color="tab:purple", lw=2, ls="--", label="Mean")
ax.set_xlabel("Log step length (m)")
ax.set_ylabel("Count")
ax.legend()
plt.tight_layout()
plt.show()

### Step lengths through time

In [ ]:
# Step length by hour of day, with the hourly median and mean overlaid
# observed data
hourly_sl_obs_median = step_df.groupby(step_df["hour_t1"].round().mod(24))["sl_"].median()
hourly_sl_obs_mean = step_df.groupby(step_df["hour_t1"].round().mod(24))["sl_"].mean()
# simulated data
hourly_sl_sim_median = sim_scattered_df.groupby(sim_scattered_df["hour_t1"].round().mod(24))["sl_"].median()
hourly_sl_sim_mean = sim_scattered_df.groupby(sim_scattered_df["hour_t1"].round().mod(24))["sl_"].mean()

fig, ax = plt.subplots(figsize=(8, 4))
# observed data
ax.scatter(step_df["hour_t1"], step_df["sl_"], s=4, alpha=0.15, color="tab:blue", label="Steps")
ax.plot(hourly_sl_obs_mean.index, hourly_sl_obs_mean.values, color="red", lw=2, marker="s", ms=4, label="Hourly mean")
ax.plot(hourly_sl_obs_median.index, hourly_sl_obs_median.values, color="black", lw=2, marker="o", ms=4, label="Hourly median")
# simulated data
ax.scatter(sim_scattered_df["hour_t1"], sim_scattered_df["sl_"], s=4, alpha=0.15, color="tab:purple", label="Steps (sim)")
ax.plot(hourly_sl_sim_mean.index, hourly_sl_sim_mean.values, color="orange", lw=2, marker="s", ms=4, label="Hourly mean (sim)")
ax.plot(hourly_sl_sim_median.index, hourly_sl_sim_median.values, color="green", lw=2, marker="o", ms=4, label="Hourly median (sim)")
ax.set_xlabel("Hour of day")
ax.set_ylabel("Step length (m)")
ax.set_xticks(range(0, 25, 3))
ax.set_xlim(0, 24)
ax.set_ylim(0, 1000)
ax.legend()
plt.tight_layout()
plt.show()

### Distribution of turning angles

In [ ]:
# Distribution of turning angles (NaN for each individual's first step, so dropped)
fig, ax = plt.subplots(figsize=(8, 4))
# observed turning angles
ax.hist(step_df["ta_"].dropna(), bins=72, density=True, range=(-np.pi, np.pi), color="tab:orange", alpha=0.5, label="Observed")
# simulated turning angles
ax.hist(sim_scattered_df["ta_"].dropna(), bins=72, density=True, range=(-np.pi, np.pi), color="tab:green", alpha=0.5, label="Simulated")
ax.set_xlabel("Turning angle (radians)")
ax.set_ylabel("Count")
ax.set_xticks([-np.pi, -np.pi / 2, 0, np.pi / 2, np.pi])
ax.set_xticklabels(["-π", "-π/2", "0", "π/2", "π"])
ax.set_xlim(-np.pi, np.pi)
plt.tight_layout()
plt.show()

### Turning angles through time

As turning angles are circular, the mean/median sit
near 0 regardless of spread. Instead, we summarise each hour with the circular SD 
and the shortest arc containing 50% of the angles (which may wrap across ±π), i.e.,
where are the densest 50% of the turning angles concentrated?

From this plot, we can see that movements are most directed (mean at 0 and most TAs
near 0) around dawn and dusk, reflecting the longer steps in those periods. 
This shows that buffalo appear to be travelling in a directed way during those periods. 

In [ ]:
# observed data
ta_obs_df = step_df.dropna(subset=["ta_"])
hourly_ta = ta_obs_df.groupby(ta_obs_df["hour_t1"].round().mod(24))["ta_"].apply(circ_summary).unstack()
# simulated data
ta_sim_df = sim_scattered_df.dropna(subset=["ta_"])
hourly_ta_sim = ta_sim_df.groupby(ta_sim_df["hour_t1"].round().mod(24))["ta_"].apply(circ_summary).unstack()

fig, (ax, ax_sd) = plt.subplots(2, 1, figsize=(8, 6), sharex=True, height_ratios=[3, 1])
ax.scatter(ta_obs_df["hour_t1"], ta_obs_df["ta_"], s=4, alpha=0.15, color="tab:orange", label="Steps")
ax.scatter(ta_sim_df["hour_t1"], ta_sim_df["ta_"], s=4, alpha=0.15, color="tab:green", label="Simulated")

offset = 0.15  # hours either side of the true hour
for summary, colour, label, shift in [
    (hourly_ta, "black", "Observed densest 50%", -offset),
    (hourly_ta_sim, "purple", "Simulated densest 50%", +offset),
]:
    for hour, row in summary.iterrows():
        # draw the arc as one segment, or two if it wraps past π
        segments = [(row.arc_start, min(row.arc_end, np.pi))]
        if row.arc_end > np.pi:
            segments.append((-np.pi, row.arc_end - 2 * np.pi))
        for lo, hi in segments:
            ax.vlines(hour + shift, lo, hi, color=colour, lw=3,
                      label=label if hour == summary.index[0] and lo == row.arc_start else None)

ax.set_ylabel("Turning angle (radians)")
ax.set_yticks([-np.pi, -np.pi / 2, 0, np.pi / 2, np.pi])
ax.set_yticklabels(["-π", "-π/2", "0", "π/2", "π"])
ax.set_ylim(-np.pi, np.pi)
ax.legend(loc="upper right")

ax_sd.plot(hourly_ta.index, hourly_ta["circ_sd"], color="black", lw=2, marker="o", ms=4)
ax_sd.plot(hourly_ta_sim.index, hourly_ta_sim["circ_sd"], color="purple", lw=2, marker="o", ms=4)
ax_sd.set_ylabel("Circular SD")
ax_sd.set_xlabel("Hour of day")
ax_sd.set_xticks(range(0, 25, 3))
ax_sd.set_xlim(0, 24)
plt.tight_layout()
plt.show()

## Create a heatmap of the simulated trajectories

Count the number of simulated locations per cell/pixel, with an option to discard the first *n* steps to reduce bias towards the initial parameters.

## `trajectory_heatmap()`
* **Inputs**
    - `trajectories`
        - simulated locations, as returned by `simulate_trajectories()` (or a list of DataFrames).
    - `transform`, `shape`
        - the landscape rasters' affine transform and `(height, width)`, so the heatmap grid shares their origin and CRS.
    - `burn_in`
        - discard the first *n* steps of every trajectory.  Early steps cluster around the release site(s) and reflect the starting conditions rather than the fitted model.
    - `agg`
        - aggregate `agg` x `agg` raster pixels into one heatmap cell.  At the native 20 m almost every occupied cell holds a single location, so some coarsening makes the surface readable.
* **Outputs**: `(counts, heatmap_transform)` — an `int32` array of locations per cell, and its affine transform.

`HEATMAP_PERIOD` selects which part of the year the counts come from: `"full"` uses
every simulated location, `"wet"` and `"dry"` keep only those whose day-of-year falls
in the November–April wet season or the May–October dry season.  The period is filtered
before `trajectory_heatmap()` is called and named in the output filenames, so the three
surfaces can be written side by side.

The result is visualised as a PNG below, written to a GeoTIFF with
`save_raster()` so it can be opened in QGIS, and added to the interactive map as
a layer that can be toggled on and off with `add_heatmap_overlay()`.

In [ ]:
HEATMAP_DF     = sim_scattered  # trajectories to summarise (sim_scatteredor sim_many)
BURN_IN        = 0              # drop the first n steps of every trajectory
HEATMAP_AGG    = 8              # heatmap cell = HEATMAP_AGG x HEATMAP_AGG raster pixels

heatmap, heatmap_transform = deepssf.trajectory_heatmap(
    HEATMAP_DF,
    raster_transform,
    landscape_tensors[0].shape,
    burn_in=BURN_IN,
    agg=HEATMAP_AGG,
)

print(f"Heatmap grid: {heatmap.shape[0]} x {heatmap.shape[1]} (n = {heatmap.shape[0] * heatmap.shape[1]:,}) cells of "
      f"{PIXEL_SIZE * HEATMAP_AGG:.0f} m")
print(f"{heatmap.sum():,} locations counted after dropping the first {BURN_IN} steps "
      f"of each trajectory")
print(f"{(heatmap > 0).sum():,} occupied cells, up to {heatmap.max():,} locations in one cell")

### Plot the heatmap
Counts are drawn on a **log** colour scale — a handful of cells in some cells usually hold an order of magnitude more locations than the rest (possibly due to bias towards the release sites or getting 'stuck' in certain areas), which
would flatten a linear scale to a single bright dot.  Empty cells are masked so
the landscape shows through.

In [ ]:
fig, ax = plt.subplots(figsize=(8.5, 7))
# Greyscale background so the (colourful) heatmap stays readable on top
rasterio.plot.show(
    landscape_tensors[0].numpy(), transform=raster_transform, ax=ax, cmap="Greys_r"
)
# ax.plot(
#     step_df["x1_"], step_df["y1_"],
#     "c.", alpha=0.15, markersize=0.5, label="observed",
# )
img = ax.imshow(
    np.ma.masked_equal(heatmap, 0),  # empty cells stay transparent
    extent=rasterio.plot.plotting_extent(heatmap, heatmap_transform),
    cmap="magma",
    norm=LogNorm(vmin=1, vmax=max(int(heatmap.max()), 2)),
    interpolation="nearest",
    alpha=0.85,
)
fig.colorbar(img, ax=ax, label=f"simulated locations per {PIXEL_SIZE * HEATMAP_AGG:.0f} m cell")
ax.set_xlabel("Easting (m, projected)")
ax.set_ylabel("Northing (m, projected)")
ax.set_title(
    f"Simulated space use ({heatmap.sum():,} locations, "
    f"steps {BURN_IN}+)"
)
plt.tight_layout()
plt.savefig(str(OUTPUT_DIR
                / f"simulated_heatmap_{heatmap.sum()}loc"
                f"_{PIXEL_SIZE * HEATMAP_AGG:.0f}m_burnin{BURN_IN}.png"), dpi=300)
plt.show()

## Save the heatmap as a raster (TIF)

## `save_raster()`
Writes any landscape-shaped array out as a GeoTIFF, given the transform and CRS
that georeference it — the heatmap here, or the habitat selection surface
above.  `nodata=0` means unvisited cells are drawn as transparent rather than as
a solid zero-valued block.

In [ ]:
# load_environmental_layers() does not return the CRS, so read it off the source
# raster the transform came from.
with rasterio.open(next(iter(LAYER_PATHS.values()))) as src:
    raster_crs = src.crs

heatmap_tif = deepssf.save_raster(
    heatmap,
    OUTPUT_DIR
    / f"simulated_heatmap_{heatmap.sum()}loc"
      f"_{PIXEL_SIZE * HEATMAP_AGG:.0f}m_burnin{BURN_IN}.tif",
    transform=heatmap_transform,
    crs=raster_crs,
    nodata=0,               # cells never visited
    band_descriptions=f"simulated locations per cell",
)

### Toggle the heatmap on the interactive map

## `add_heatmap_overlay()`
`plot_trajectories_folium()` returns the `folium.Map`, so the heatmap can be
added to it afterwards as an image overlay.  folium writes each layer's
JavaScript in the order it was added, so `add_heatmap_overlay()` moves the layer
control back to the end — otherwise it would run before the overlay exists and
throw, taking the rest of the map's script with it.  The heatmap then picks up
its own checkbox alongside the observed locations and each simulated trajectory.

The counts are warped from the projected CRS to Web Mercator — the projection
Leaflet stretches an overlay in — and coloured with the same log scale as the
PNG, with cells below `vmin` (1 by default) left fully transparent.

In [ ]:
# add some of the simulated trajectories to the map
heat_ids = HEATMAP_DF["trajectory_id"].unique()[:3]

fmap_heat = deepssf.plot_trajectories_folium(
    HEATMAP_DF[HEATMAP_DF["trajectory_id"].isin(heat_ids)],
    observed=step_df,
    src_crs="EPSG:3112",
    basemap="esri_imagery",
    zoom_start=11,
    max_observed_points=100000,
)

deepssf.add_heatmap_overlay(
    fmap_heat,
    heatmap,
    heatmap_transform,
    src_crs="EPSG:3112",
    name=f"heatmap, (max {heatmap.max():,} per cell)",
    cmap="magma",
    opacity=0.75,
)

heatmap_html = OUTPUT_DIR / "simulated_heatmap_map.html"
fmap_heat.save(str(heatmap_html))
print(f"Map saved: {heatmap_html}")
fmap_heat

# Estimating landscape connectivity with betweenness

Such as from:

Bastille-Rousseau, G., Douglas-Hamilton, I., Blake, S., Northrup, J. M., & Wittemyer, G. (2018). Applying network theory to animal movements to identify properties of landscape space use. Ecological Applications: A Publication of the Ecological Society of America, 28(3), 854–864. https://doi.org/10.1002/eap.1697

and

Hofmann, D. D., Cozzi, G., McNutt, J. W., Ozgul, A., & Behr, D. M. (2023). A three-step approach for assessing landscape connectivity via simulated dispersal: African wild dog case study. Landscape Ecology, 38(4), 981–998. https://doi.org/10.1007/s10980-023-01602-4

## Betweenness

Quantifies the importance of different elements of the matrix in facilitating connectivity between other patches or habitats within the landscape. It identifies key corridors or bottlenecks that serve as critical links for maintaining overall connectivity in the landscape. It is calculated as the number of times the simulated individuals enter each raster cell (which are typically aggregated to a coarser scale than the spatial covariates used in the simulations). Betweenness measures the importance of each cell within the landscape for maintaining functional connectivity

Adapted from: https://github.com/DavidDHofmann/DispersalSimulation/blob/main/AWDCaseStudy/02_Analysis/09_Betweenness.R

## How it is calculated here

Following the R script, in four steps:

1. **A coarse grid of nodes.**  The landscape rasters are 20 m, much finer than the scale
   connectivity is meaningful at, so they are aggregated by `BETWEENNESS_AGG` into 500 m
   cells.  Each cell is one node in the network.  The grid keeps the rasters' origin and
   CRS, so it lines up with the covariates and with the heatmap above.
2. **Transitions become edges.**  Every simulated path is walked cell by cell, and each
   move from one cell into the next is an edge.  A trajectory contributes a given
   `(from, to)` pair *once*, however often it repeats it (`singlecount = TRUE` in the R
   script), so the pooled count is "how many of the simulated pigs ever made this move",
   not "how many times it happened".
3. **Edge weights invert the counts**: `weight = mean(count) / count`.  networkx reads the
   weight as a *cost*, so a transition many pigs made is cheap and shortest paths follow
   the routes the simulation actually took.
4. **Betweenness** of a node is the number of those shortest paths running through it.
   Cells with high betweenness are the ones movement is funnelled through.

### `path_cells()` and `transition_counts()`
* **Inputs**
    - `BETWEENNESS_DF`
        - simulated locations, as returned by `simulate_trajectories()` — the same input
          `trajectory_heatmap()` takes.
    - `BETWEENNESS_AGG`
        - aggregate `agg` x `agg` raster pixels into one network cell.  Betweenness is an
          all-pairs shortest path calculation, so the node count sets the runtime: 500 m
          cells give 3,672 nodes and about a minute, 250 m cells give 15,651 and about
          twenty.
    - `BETWEENNESS_EPS`
        - spacing, in cell widths, at which each step is densified before the cells under
          it are read off.  Simulated steps have a median length of ~26 m but reach
          ~700 m, so a long step can jump clean over a 500 m cell; densifying catches it.
    - `BETWEENNESS_BURN_IN`, `BETWEENNESS_MAX_STEPS`
        - drop the first *n* steps of every trajectory, then keep at most *m* of what is
          left.
    - `BETWEENNESS_K`
        - `None` computes betweenness exactly; an integer estimates it from that many
          randomly chosen pivot nodes, which is what makes a finer grid tractable.
* **Outputs**: `betweenness` — a `float32` array on the coarsened grid, plus its affine
  transform `bet_transform`, ready for `save_raster()` and `add_heatmap_overlay()` just
  like the heatmap.

**`BETWEENNESS_MAX_STEPS`**  Given enough steps every
trajectory eventually visits most of the landscape, every adjacent pair of cells picks up
an edge from nearly every trajectory, the weights flatten towards uniform, and betweenness
ends up describing the *grid* rather than the *movement*. The R script on David's GitHub handles this by recomputing the map at several
step budgets (`design$steps = c(125, 500, 2000)`) and checking the pattern has settled;
the same is worth doing here.

In [ ]:
BETWEENNESS_DF        = sim_scattered  # trajectories to build the network from
BETWEENNESS_AGG       = 8              # network cell = AGG x AGG raster pixels
BETWEENNESS_BURN_IN   = 0              # drop the first n steps of every trajectory
BETWEENNESS_MAX_STEPS = 10_000         # ...and keep at most this many of what is left
BETWEENNESS_EPS       = 0.25           # densify each step to this spacing, in cell widths
BETWEENNESS_K         = None           # None = exact; an int samples that many pivot nodes

# The network grid: the landscape rasters' origin and CRS with cells AGG times larger.
# ceil() rather than round() so the grid covers the whole raster — the same arithmetic
# trajectory_heatmap() uses, which is what makes the two grids line up cell for cell.
bet_height = math.ceil(landscape_tensors[0].shape[0] / BETWEENNESS_AGG)
bet_width  = math.ceil(landscape_tensors[0].shape[1] / BETWEENNESS_AGG)
bet_transform = raster_transform * Affine.scale(BETWEENNESS_AGG)
N_NODES = bet_height * bet_width
BET_CELL_M = PIXEL_SIZE * BETWEENNESS_AGG

def path_cells(x, y, eps=BETWEENNESS_EPS):
    """The sequence of network cells one simulated path passes through.

    Points outside the raster come back as -1 rather than being dropped, so that
    a pig which leaves the landscape and returns does not look like one long
    move between the cells either side of the excursion.
    """
    col, row = ~bet_transform * (np.asarray(x, dtype=float), np.asarray(y, dtype=float))
    if col.size < 2:
        return np.empty(0, dtype=np.int64)

    # Densify each step into enough sub-points that none can skip over a cell.
    # Most steps are far shorter than a cell and get n_sub = 1, so this costs
    # little; the occasional ~700 m step is the reason it is here at all.
    dcol, drow = np.diff(col), np.diff(row)
    n_sub = np.maximum(1, np.ceil(np.hypot(dcol, drow) / eps)).astype(np.int64)
    seg   = np.repeat(np.arange(n_sub.size), n_sub)      # step each sub-point belongs to
    start = np.repeat(np.cumsum(n_sub) - n_sub, n_sub)   # index of that step's first one
    frac  = (np.arange(n_sub.sum()) - start) / np.repeat(n_sub, n_sub)
    # Each step contributes its start and the points along it; the very last
    # location is the start of no step, so it is appended by hand.
    col_i = np.append(col[seg] + frac * dcol[seg], col[-1])
    row_i = np.append(row[seg] + frac * drow[seg], row[-1])

    c = np.floor(col_i).astype(np.int64)
    r = np.floor(row_i).astype(np.int64)
    inside = (r >= 0) & (r < bet_height) & (c >= 0) & (c < bet_width)
    node = np.where(inside, r * bet_width + c, -1)   # node ids run in raster order
    # Consecutive sub-points usually land in the same cell.  Keeping only the
    # changes leaves one entry per cell entered, and no from == to self-loops.
    return node[np.r_[True, node[1:] != node[:-1]]]


def transition_counts(df, id_col="trajectory_id"):
    """Pool cell-to-cell transitions over every trajectory.

    A trajectory contributes each (from, to) pair once however often it repeats
    it, so the counts returned are the number of *trajectories* that ever made
    each move.  Trajectories are handled one at a time rather than as a single
    array: at 599 x 100,000 locations the intermediate keys would otherwise run
    to several GB.
    """
    keys, n_edge_drops = [], 0
    for _tid, sub in df.groupby(id_col, sort=True):
        node = path_cells(sub["x"].to_numpy(), sub["y"].to_numpy())
        if node.size < 2:
            continue
        src, dst = node[:-1], node[1:]
        ok = (src >= 0) & (dst >= 0)          # drop moves touching the outside
        n_edge_drops += int((~ok).sum())
        keys.append(np.unique(src[ok] * N_NODES + dst[ok]))
    if not keys:
        raise ValueError(
            "no transitions — no trajectory entered more than one network cell"
        )
    key, count = np.unique(np.concatenate(keys), return_counts=True)
    return key // N_NODES, key % N_NODES, count, n_edge_drops


print(f"Network grid: {bet_height} x {bet_width} = {N_NODES:,} cells of {BET_CELL_M:.0f} m")

### Build the network and calculate betweenness
The period filter is the same `filter_period()` the heatmap uses, so a betweenness
surface and a heatmap built with the same `PERIOD` describe the same slice of the year.

This chunk can take some time to run!

In [ ]:
bet_df = BETWEENNESS_DF[
    (BETWEENNESS_DF["step"] >= BETWEENNESS_BURN_IN)
    & (BETWEENNESS_DF["step"] < BETWEENNESS_BURN_IN + BETWEENNESS_MAX_STEPS)
]
if bet_df.empty:
    raise ValueError(
        f"no simulated locations left — steps {BETWEENNESS_BURN_IN} to "
        f"{BETWEENNESS_BURN_IN + BETWEENNESS_MAX_STEPS}"
    )

# Reused by the plot, GeoTIFF and web map below so they all describe the same subset.
N_BET_TRAJ = bet_df["trajectory_id"].nunique()

t0 = time.time()
bet_src, bet_dst, bet_count, n_edge_drops = transition_counts(bet_df)
print(f"Period: {len(bet_df):,} locations from {N_BET_TRAJ} "
      f"trajectories, steps {BETWEENNESS_BURN_IN} to "
      f"{BETWEENNESS_BURN_IN + BETWEENNESS_MAX_STEPS - 1}")
print(f"{len(bet_src):,} unique transitions found in {time.time() - t0:.0f} s "
      f"({n_edge_drops:,} dropped at the raster edge)")

# networkx reads the weight as a cost, so invert the counts: a transition many
# trajectories made is cheap, and shortest paths prefer it.
bet_weight = bet_count.mean() / bet_count

# Every cell is a node, including the ones no trajectory reached — they sit in the
# network isolated and score zero, as `vertices = ver` does in the R script.
bet_net = nx.DiGraph()
bet_net.add_nodes_from(range(N_NODES))
bet_net.add_weighted_edges_from(
    zip(bet_src.tolist(), bet_dst.tolist(), bet_weight.tolist(), strict=True)
)
print(f"Network: {bet_net.number_of_nodes():,} nodes, {bet_net.number_of_edges():,} edges")

t0 = time.time()
bet_scores = nx.betweenness_centrality(
    bet_net, k=BETWEENNESS_K, weight="weight", normalized=False, seed=1
)
print(f"Betweenness computed in {time.time() - t0:.0f} s"
      + ("" if BETWEENNESS_K is None else f" (approximate, {BETWEENNESS_K} pivots)"))

# Cells no trajectory ever entered are left as NaN rather than 0, so the map can
# tell them apart from cells that are in the network but on no shortest path.
in_network = np.zeros(N_NODES, dtype=bool)
in_network[bet_src] = True
in_network[bet_dst] = True
betweenness = np.full(N_NODES, np.nan, dtype=np.float32)
betweenness[in_network] = [bet_scores[n] for n in np.flatnonzero(in_network)]
betweenness = betweenness.reshape(bet_height, bet_width)

print(f"{in_network.sum():,} of {N_NODES:,} cells were entered by at least one "
      f"trajectory; betweenness runs from 0 to {np.nanmax(betweenness):,.0f}")

### Plot the betweenness surface
Drawn on a **square-root** scale, as in the R script — a handful of corridors carry
far more shortest paths than anywhere else, and on a linear scale everything but those
few cells goes black.  Cells no trajectory entered are masked so the landscape shows
through, the same way empty cells are masked on the heatmap.

In [ ]:
fig, ax = plt.subplots(figsize=(8.5, 7))
# Greyscale background so the (colourful) betweenness stays readable on top
rasterio.plot.show(
    landscape_tensors[0].numpy(), transform=raster_transform, ax=ax, cmap="Greys_r"
)
img = ax.imshow(
    np.ma.masked_invalid(np.sqrt(betweenness)),  # unentered cells stay transparent
    extent=rasterio.plot.plotting_extent(betweenness, bet_transform),
    cmap="magma",
    interpolation="nearest",
    alpha=0.85,
)
fig.colorbar(img, ax=ax, label=f"sqrt(betweenness) per {BET_CELL_M:.0f} m cell")
ax.set_xlabel("Easting (m, projected)")
ax.set_ylabel("Northing (m, projected)")
ax.set_title(
    f"Landscape connectivity ({N_BET_TRAJ} trajectories, "
    f"{BETWEENNESS_MAX_STEPS:,} steps)"
)
plt.tight_layout()
plt.savefig(str(OUTPUT_DIR
                / f"betweenness_{N_BET_TRAJ}traj_{BET_CELL_M:.0f}m"
                f"_steps{BETWEENNESS_MAX_STEPS}.png"), dpi=300)
plt.show()

### Save the betweenness surface as a raster (TIF)
Same `save_raster()` as the heatmap, on the coarsened transform.  `nodata=np.nan` marks
the cells no trajectory entered, so they open as transparent rather than as real zeros
next to cells that genuinely scored zero.

In [ ]:
betweenness_tif = deepssf.save_raster(
    betweenness,
    OUTPUT_DIR
    / f"betweenness_{N_BET_TRAJ}traj_{BET_CELL_M:.0f}m"
      f"_steps{BETWEENNESS_MAX_STEPS}.tif",
    transform=bet_transform,
    crs=raster_crs,
    nodata=np.nan,          # cells no trajectory entered
    band_descriptions="betweenness per cell",
)

### Toggle betweenness on the interactive map
`add_heatmap_overlay()` takes any array on a regular grid, not just counts, so the
betweenness surface goes onto a folium map exactly as the heatmap did — on the same
square-root scale as the PNG above.

In [ ]:
bet_ids = bet_df["trajectory_id"].unique()[:3]

fmap_bet = deepssf.plot_trajectories_folium(
    bet_df[bet_df["trajectory_id"].isin(bet_ids)],
    observed=step_df,
    src_crs="EPSG:3112",
    basemap="esri_imagery",
    zoom_start=11,
    max_observed_points=100000,
)

# add_heatmap_overlay() warps the array with rasterio and has no NaN handling, so the
# unentered cells go back to 0 and are hidden by vmin instead.  sqrt rather than the
# function's log scale, to match the PNG.
deepssf.add_heatmap_overlay(
    fmap_bet,
    np.sqrt(np.nan_to_num(betweenness, nan=0.0)),
    bet_transform,
    src_crs="EPSG:3112",
    name=f"betweenness (max {np.nanmax(betweenness):,.0f})",
    cmap="magma",
    log_scale=False,
    vmin=1,
    opacity=0.95,
)

betweenness_html = OUTPUT_DIR / f"betweenness_map.html"
fmap_bet.save(str(betweenness_html))
print(f"Map saved: {betweenness_html}")
fmap_bet